# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We construct numerical features from raw Search Console metrics (impressions, clicks, ctr, and position). Categorical URLs are encoded as length/structure properties, missing values are filled with column medians, and ratio features are calculated safely without division-by-zero errors.

In [8]:
import os
import numpy as np
import pandas as pd
import urllib.request

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------

data_path = "content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    urllib.request.urlretrieve(url, data_path)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


# --------------------------------------------------
# 2. Create target using available click columns
# --------------------------------------------------

# Current 30-day clicks vs previous 30-day clicks
df["click_drop_pct"] = (
    (df["clicks_prev_30d"] - df["clicks_last_30d"])
    / (df["clicks_prev_30d"] + 1e-5)
)

# Total clicks
df["total_clicks"] = (
    df["clicks_prev_30d"].fillna(0)
    + df["clicks_last_30d"].fillna(0)
)

# Content needs refresh if clicks dropped by more than 20%
df["needs_refresh"] = (
    df["click_drop_pct"] > 0.20
).astype(int)


# --------------------------------------------------
# 3. Engineer features
# --------------------------------------------------

# Log impressions
df["log_impressions"] = np.log1p(
    df["impressions_90d"].fillna(0)
)

# Log clicks
df["log_clicks"] = np.log1p(
    df["total_clicks"].fillna(0)
)

# CTR
df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df["ctr"] = df["ctr"].fillna(
    df["ctr"].median()
)

# Average position
df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

df["avg_position"] = df["avg_position"].fillna(
    df["avg_position"].median()
)

# URL is NOT present in this dataset.
# Instead, use content characteristics.
df["word_count"] = pd.to_numeric(
    df["word_count"], errors="coerce"
).fillna(0)

df["char_count"] = pd.to_numeric(
    df["char_count"], errors="coerce"
).fillna(0)

# Content age
df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
).fillna(0)

# Engagement rate
df["engagement_rate"] = pd.to_numeric(
    df["engagement_rate"], errors="coerce"
).fillna(0)

# AI traffic
df["ai_traffic_pct"] = pd.to_numeric(
    df["ai_traffic_pct"], errors="coerce"
).fillna(0)


# --------------------------------------------------
# 4. Build feature vector X
# --------------------------------------------------

feature_columns = [
    "log_impressions",
    "log_clicks",
    "ctr",
    "avg_position",
    "word_count",
    "char_count",
    "content_age_days",
    "engagement_rate",
    "ai_traffic_pct"
]

X = df[feature_columns].copy()

y = df["needs_refresh"].copy()


# --------------------------------------------------
# 5. Clean feature vector
# --------------------------------------------------

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)


# --------------------------------------------------
# 6. Display result
# --------------------------------------------------

print("\nFeature Vector Shape:", X.shape)

print("\nFeature Columns:")
print(X.columns.tolist())

print("\nFirst 5 rows of Feature Vector:")
print(X.head())

print("\nTarget Distribution:")
print(y.value_counts())

print("\nTarget Percentage:")
print(y.value_counts(normalize=True) * 100)

Dataset loaded successfully!
Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Feature Vector Shape: (30000, 9)

Feature Columns:
['log_impressions', 'log_clicks', 'ctr', 'avg_position', 'word_count', 'char_count', 'content_age_days', 'engage

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


log_impressions: Log-transformed search views prior to the prediction timestamp. Missing values filled with 0. Available before prediction window.

log_clicks: Log-transformed organic clicks prior to prediction. Missing values filled with 0. Available before prediction window.

ctr: Click-through rate calculated prior to prediction. Missing values filled with median. Available before prediction window.

position: Average ranking position prior to prediction. Missing values filled with median. Available before prediction window.

url_length: Character length of URL string. Missing values filled with empty string. Available before prediction window.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We verify that no engineered feature perfectly correlates (|r| > 0.95) with the target label needs_refresh and that no target-derived metric (click_drop_pct) is passed directly into the training matrix.

In [9]:
# ============================================================
# 3. THE LEAKAGE HUNT
# ============================================================

print("=" * 60)
print("LEAKAGE HUNT")
print("=" * 60)

# ------------------------------------------------------------
# TEST 1: Check whether target-derived columns are in X
# ------------------------------------------------------------

target_derived_columns = [
    "click_drop_pct",
    "total_clicks",
    "needs_refresh"
]

leaking_columns = [
    col for col in X.columns
    if col in target_derived_columns
]

print("\n[TEST 1] Target-derived columns in X:")
print(leaking_columns)

assert len(leaking_columns) == 0, (
    f"LEAKAGE DETECTED: {leaking_columns} are directly derived "
    "from the target."
)

print("PASS: No directly target-derived columns found.")


# ------------------------------------------------------------
# TEST 2: Check suspicious click-based features
# ------------------------------------------------------------

click_based_features = [
    col for col in X.columns
    if "click" in col.lower()
]

print("\n[TEST 2] Click-related features in X:")
print(click_based_features)

if click_based_features:
    print(
        "WARNING: These features use click information. "
        "Because the target is defined using click change, "
        "they may contain target information."
    )


# ------------------------------------------------------------
# TEST 3: Correlation with target
# ------------------------------------------------------------

corr_matrix = X.assign(target=y).corr(numeric_only=True)

target_corr = (
    corr_matrix["target"]
    .drop("target")
    .sort_values(ascending=False)
)

print("\n[TEST 3] Feature-Target Correlation:")
print(target_corr)

max_corr = target_corr.abs().max()

print(
    f"\nMaximum Feature-Target Correlation: "
    f"{max_corr:.4f}"
)

if max_corr >= 0.95:
    print(
        "WARNING: Very high correlation detected. "
        "Investigate these features for leakage."
    )
else:
    print(
        "PASS: No feature has extremely high "
        "linear correlation with the target."
    )


# ------------------------------------------------------------
# TEST 4: Check original target columns
# ------------------------------------------------------------

target_source_columns = [
    "clicks_prev_30d",
    "clicks_last_30d"
]

features_using_target_sources = [
    col for col in X.columns
    if col in target_source_columns
]

print("\n[TEST 4] Raw target-source columns in X:")
print(features_using_target_sources)

assert len(features_using_target_sources) == 0, (
    "LEAKAGE DETECTED: Raw columns used to construct "
    "the target are present in X."
)

print("PASS: Raw target-source columns are not directly in X.")


# ------------------------------------------------------------
# TEST 5: Product / label flags
# ------------------------------------------------------------

suspicious_flags = [
    col for col in X.columns
    if any(word in col.lower() for word in [
        "label",
        "target",
        "refresh",
        "flag",
        "decision",
        "recommendation"
    ])
]

print("\n[TEST 5] Suspicious product/label flags:")
print(suspicious_flags)

if suspicious_flags:
    print(
        "WARNING: Review these columns manually "
        "before model training."
    )
else:
    print("PASS: No obvious product/label flags found.")


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LEAKAGE HUNT COMPLETED")
print("=" * 60)

LEAKAGE HUNT

[TEST 1] Target-derived columns in X:
[]
PASS: No directly target-derived columns found.

[TEST 2] Click-related features in X:
['log_clicks']

[TEST 3] Feature-Target Correlation:
log_clicks          0.365708
log_impressions     0.332082
engagement_rate     0.058107
ctr                 0.038375
word_count          0.036316
char_count          0.030620
ai_traffic_pct     -0.010780
content_age_days   -0.021432
avg_position       -0.095203
Name: target, dtype: float64

Maximum Feature-Target Correlation: 0.3657
PASS: No feature has extremely high linear correlation with the target.

[TEST 4] Raw target-source columns in X:
[]
PASS: Raw target-source columns are not directly in X.

[TEST 5] Suspicious product/label flags:
[]
PASS: No obvious product/label flags found.

LEAKAGE HUNT COMPLETED


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

click_drop_pct: Excluded because it is used directly to derive the target label needs_refresh (direct target leakage).

Future Post-Refresh Clicks / Traffic: Excluded because metrics occurring after the prediction point do not exist at inference time.

Raw url Strings: Excluded to protect domain privacy and prevent high-cardinality overfitting.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.